# Federated learning with Flower (simulation fallback)

Five clients train together a model that recognises clothes and shoes (Fashion-MNIST,
10 classes), **without ever sharing their images**. In each round, the server sends the
model to the clients, each one trains it on its own data and sends back only the
weights, then the server averages them.

The clients are simulated on this machine, each in its own process with its own data.
The code is the same as for a real deployment across several machines.

This notebook **launches** the experiments. The results show up in the **dashboard**:
open it in a tab next to this one and keep it open, it updates every round.

| Dashboard tab | What it shows |
|---|---|
| **Solo or together?** | The federated model versus each client trained alone, and versus fully centralised training |
| **The run** | The local models in each round, who took part, client drift |
| **Network** | The messages exchanged during a round, replayed as an animation |
| **Compare** | Several runs on one chart |

A 10-round run takes 1 to 2 minutes; the first round takes 20 to 60 s, while the
clients start.

## Setup

In [ ]:
from flworkshop import viz

viz.setup()

## 1. Solo or together?

Each client holds only **2 classes** out of 10: one has only T-shirts and sneakers,
another only trousers and shirts, and so on. Launch the federated run, then open the
**Solo or together?** tab. The "solo" and "centralised" references were computed in
advance with exactly the same data and the same training budget.

In [ ]:
viz.run("solo_vs_together", partitioning="pathological")

> **Questions**
> 1. How many classes does a client recognise alone? And the federated model?
> 2. In the *What each model can recognise* table, two classes are recognised by
>    nobody, not even the centralised model. Why? (Look at *Each client's data*.)
> 3. What did the clients share to get this result?

## 2. What travels during a round

Open the **Network** tab on the same run. Each round is replayed as an animation, with
its real timeline and the volume exchanged. The slider lets you replay any round.

In [ ]:
viz.open("solo_vs_together", view="network")

The same exchanges, seen in the Flower server log:

In [ ]:
viz.log("solo_vs_together")

> **Questions**
> 1. In which order do the messages travel? Does a client ever talk directly to another
>    client?
> 2. How many MB travelled in 10 rounds? Compare with the volume of the clients' data.
> 3. Find `Sampled`, `Received` and `Aggregated` in the log: which step of the animation
>    does each line match?

## 3. A more realistic imbalance

With a **Dirichlet** partitioning (α = 0.1), each client mostly holds a few classes, and
the amounts of data differ a lot between clients.

In [ ]:
viz.run("dirichlet", partitioning="dirichlet")

> **Questions**
> 1. *Solo or together?* tab: is the gap between solo and together as large as in 1?
>    Why?
> 2. *Network* tab, timeline: why does everyone wait for the same client?
> 3. *The run* tab: which client moves furthest from the global model (*Client drift*)?
>    What is special about its data?

## 4. When the data look alike

With **IID** data, each client holds a representative sample of all classes.

In [ ]:
viz.run("iid", partitioning="iid")

> **Questions**
> 1. Does a client alone do well? What does federation still bring?
> 2. *Compare* tab: rank the three runs. What makes federated learning hard?

## 5. Partial participation and dropouts

In practice, only part of the clients are asked to train in each round, and some do not
finish. Follow these two runs in **The run** tab.

**a.** Only 2 clients out of 5, sampled at random, train in each round:

In [ ]:
viz.run("participation_40", partitioning="dirichlet", fraction_train=0.4, view="run")

**b.** Each client has a one-in-two chance of disconnecting during a round:

In [ ]:
viz.run("dropouts_50", partitioning="dirichlet", dropout_rate=0.5, view="run")

> **Questions**
> 1. Does federation stop when a client drops out? What does the server do with the
>    missing ones? (*Network* tab on `dropouts_50`.)
> 2. Why does the global model curve go up in a saw-tooth? Look at who takes part in
>    each round and which classes they hold.
> 3. *Compare* tab with `dirichlet`: why is the last-round accuracy no longer enough to
>    compare two runs?

## Going further

**Harder images: CIFAR-10** (planes, cars, animals…)

In [ ]:
# viz.run("cifar_solo_vs_together", dataset="cifar10", partitioning="pathological")

**FedProx** penalises the distance between the local model and the global model
(`proximal_mu`). Does it beat FedAvg here? Look at *Client drift* in *The run* tab.

In [ ]:
# viz.run("fedprox", partitioning="dirichlet", proximal_mu=0.1, view="run")

## Your turn

Every option below is passed to `flwr run --run-config`: the app code never changes.

| Key | Default | Role |
|---|---|---|
| `partitioning` | `"iid"` | `"iid"`, `"dirichlet"` or `"pathological"` |
| `dirichlet_alpha` | 0.1 | Smaller = more unbalanced |
| `classes_per_partition` | 2 | Classes per client with `"pathological"` |
| `fraction_train` | 1.0 | Share of clients sampled in each round |
| `dropout_rate` | 0.0 | Probability that a client drops out during a round |
| `num_server_rounds` | 10 | Number of rounds |
| `local_epochs` | 2 | Local training epochs per round |
| `learning_rate` | 0.01 | Client learning rate |
| `proximal_mu` | 0.0 | > 0 enables FedProx |
| `dataset` | `"fashion_mnist"` | `"fashion_mnist"` or `"cifar10"` |

The "solo / centralised" references exist only for the exercise configurations;
elsewhere, the *Solo or together?* tab shows the federated model alone.
`viz.stop()` stops the run in progress.

In [ ]:
# viz.run("my_try", partitioning="pathological", classes_per_partition=3)